# **ENCODER DECODER USING NLP**

In [ ]:
import numpy as np
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Data
eng = ["i am happy", "i am sad"]
hin = ["start main khush hun end", "start main udaas hun end"]

# Tokenize + pad
et, ht = Tokenizer(), Tokenizer(filters='')
et.fit_on_texts(eng); ht.fit_on_texts(hin)

e = pad_sequences(et.texts_to_sequences(eng))
h = pad_sequences(ht.texts_to_sequences(hin))

# Target shift
y = np.zeros((h.shape[0], h.shape[1], 1))
y[:, :-1, 0] = h[:, 1:]

# Model
ei, di = Input((None,)), Input((None,))
_, h1, c1 = LSTM(64, return_state=True)(Embedding(len(et.word_index)+1,64)(ei))
do, _, _ = LSTM(64, return_sequences=True, return_state=True)(
    Embedding(len(ht.word_index)+1,64)(di), initial_state=[h1,c1])
out = Dense(len(ht.word_index)+1, activation='softmax')(do)

m = Model([ei, di], out)
m.compile('adam', 'sparse_categorical_crossentropy')
m.fit([e,h], y, epochs=200, verbose=0)

# Test
test = pad_sequences(et.texts_to_sequences(["i am happy"]), maxlen=e.shape[1])
pred = m.predict([test, h[:1]], verbose=0)

rev = {v:k for k,v in ht.word_index.items()}
words = []

for i in pred[0]:
    idx = np.argmax(i)
    word = rev.get(idx, '')

    if word not in ['start', 'end']:
        words.append(word)

print("Output:", " ".join(words))

Output: main khush hun 
